In [ ]:
import os
import geopandas as gpd
import glob
from pyproj import CRS
from shapely.ops import unary_union

wkt = """PROJCRS["TMCI-5.5",
    BASEGEOGCRS["ITRF2014",
        DATUM["International Terrestrial Reference Frame 2014",
            ELLIPSOID["IAG-GRS80",6378137,298.257222101,
                LENGTHUNIT["metre",1]],
            ID["EPSG",1165]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]]],
    CONVERSION["unnamed",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",-5.5,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",0.9996,
            SCALEUNIT["unity",1],
            ID["EPSG",8805]],
        PARAMETER["False easting",500000,
            LENGTHUNIT["metre",1],
            ID["EPSG",8806]],
        PARAMETER["False northing",0,
            LENGTHUNIT["metre",1],
            ID["EPSG",8807]]],
    CS[Cartesian,2],
        AXIS["easting",east,
            ORDER[1],
            LENGTHUNIT["metre",1]],
        AXIS["northing",north,
            ORDER[2],
            LENGTHUNIT["metre",1]],
    USAGE[
        AREA["Between 8.61°W and 2.5°W, northern hemisphere between equator and 84°N."],
        BBOX[0,-8.61,84,-2.5],ID["EPSG":336295]]]"""

custom_crs = CRS.from_wkt(wkt)

sp_shape = gpd.read_file("/home/userpc/PRESFOR/REGION_TONKPI/shapefiles_tonkpi_tmci/limites_sous_prefecture_tonkpi__tmci55.shp").to_crs(custom_crs)
villages = gpd.read_file("/home/userpc/PRESFOR/REGION_TONKPI/shapefiles_tonkpi_tmci/villages_delimites_tonkpi__tmci55.shp").to_crs(custom_crs)
files = glob.glob("/home/userpc/PRESFOR/REGION_TONKPI/shapefiles_tonkpi_tmci_/*.shp")

for i in range(0,33):
    dir = sp_shape['SOUS_PREF'][i]
    try:
        os.mkdir(f"/home/userpc/PRESFOR/REGION_TONKPI/shapefiles_tonkpi_sp_new/{dir}")
    except FileExistsError:
        pass

    gdf_sp = sp_shape.loc[[i]]
    gdf_v = villages[villages["NOM_SSPREF"] == dir]
    gdf_v.to_file(f"/home/userpc/PRESFOR/REGION_NAWA/SHAPEFILES_SP_NAWA/{dir}/villages_delimites_sp_{dir.lower()}__tmci55.shp")
    gdf_sp.to_file(f"/home/userpc/PRESFOR/REGION_TONKPI/shapefiles_tonkpi_sp_new/{dir}/sp_{dir.lower()}.shp")
    union_v = unary_union(gdf_v.geometry)
    union_buffered = union_v.buffer(100)
    
    for file_ in files :
            shape = gpd.read_file(file_)
            shape = shape.to_crs(custom_crs)
            shape_point = shape[shape.geometry.type == 'POINT']
            if not shape_point.empty:
                shape["touches_buffered"] = shape.geometry.apply(lambda geom : geom.within(union_buffered))
            else:
                shape["touches_buffered"] = shape.geometry.apply(lambda geom : geom.intersects(union_buffered))
            res_int = shape[shape["touches_buffered"]]
            if not res_int.empty :
                res_int.to_file(f"/home/userpc/PRESFOR/REGION_TONKPI/shapefiles_tonkpi_sp_new/{dir}/{file_.split('.')[0].split('/')[-1]}_sp_{dir.lower()}.shp")
